# Ellipsoids

## Objectives

This tutorial introduces the Eshelby inhomogeneity problem and the associated concentration tensors. It shows how to build an ellipsoidal inhomogeneity in *Echoes* and how to compute the four concentration tensors relating microscopic and macroscopic strains and stresses.

## Imports

In [ ]:
import numpy as np
from echoes import *
import math

np.set_printoptions(precision=8, suppress=True)
# to display only 8 significant digits of array components

## The Eshelby inhomogeneity problem [@eshelby1957]

As in the inclusion problem presented in [§], the studied domain $\Omega$ is the whole $\mathbb{R}^3$ space. An ellipsoidal subdomain is still defined by $\underline{x}\cdot({}^{t}{\boldsymbol{A}}\cdot\boldsymbol{A})^{-1}\cdot\underline{x}\leq 1$.

The material still obeys a linear elastic behavior with a uniform stiffness tensor $\mathbb{C}$ outside the ellipsoid but now the material filling the ellipsoid is characterized by another stiffness tensor $\mathbb{C}^\mathcal{I}$ without any polarization. This situation somehow corresponds to a fictitious polarization such that

$$
    \boldsymbol{\sigma}(\underline{x})=\mathbb{C}:\boldsymbol{\varepsilon}(\underline{x})+\boldsymbol{\tau}(\underline{x})\quad\textrm{where}\quad
    \boldsymbol{\tau}(\underline{x})=
    \left\{
    \begin{array}{ll}
    \boldsymbol{0}&\textrm{ if }\underline{x} \notin \mathcal{E}_{\boldsymbol{A}}\\
    \left(\mathbb{C}^\mathcal{I}-\mathbb{C}\right):\boldsymbol{\varepsilon}(\underline{x})=\delta\mathbb{C}:\boldsymbol{\varepsilon}(\underline{x})  &\textrm{ if }\underline{x} \in \mathcal{E}_{\boldsymbol{A}}
    \end{array}
    \right.
$$

In addition the displacement is now of the form $\boldsymbol{E}\cdot\underline{x}$ at infinity (i.e. remote homogeneous strain condition such that the problem solution would be $\boldsymbol{\varepsilon}=\boldsymbol{E}$ if the ellipsoid and the matrix shared the same elasticity).

![](../img/eshelbypb.png){width=30%}

Eshelby inhomogeneity problem

The second important result derived in [@eshelby1957] is that the strain tensor field solution to this problem is also uniform within the ellipsoidal domain (not outside!) and writes

$$
   \forall \underline{x} \in \mathcal{E}_{\boldsymbol{A}},\quad \boldsymbol{\varepsilon}(\underline{x})=\mathbb{A}^E:\boldsymbol{E} \quad\textrm{ where }
   \mathbb{A}^E=\left(\mathbb{I}+\mathbb{P}:\delta\mathbb{C}\right)^{-1}
$$

$\mathbb{A}^E$ is called the **strain-strain concentration tensor** for the Eshelby problem and $\mathbb{P}(\boldsymbol{A},\mathbb{C})$ is the Hill polarization tensor already introduced in [§], depending only on the shape and orientation of the ellipsoid $\boldsymbol{A}$ and the matrix behavior $\mathbb{C}$, not on that of the ellipsoidal inhomogeneity $\mathbb{C}^\mathcal{I}$.

## Ellipsoidal inhomogeneity in *Echoes*

An inhomogeneity of ellipsoidal shape is built in *Echoes* by

```
ell = ellipsoid(shape=..., prop={"C":..., "D":...})
```

Such an `ellipsoid` object is defined by a `shape` (either `ellipsoidal`, `spheroidal` or `spherical` as presented in [§]) and a dictionary of properties of names arbitrarily chosen by the user.

It is possible to change the `shape` and `prop` values after construction:

```
ell.shape = spheroidal(0.1, math.pi/3, math.pi/4)
ell.set_prop("C", stiff_kmu(3., 2.))
```

In order to simulate the Eshelby problem so as to calculate the concentration tensor $\mathbb{A}^E$, it is necessary to define the reference medium in which the ellipsoid is embedded. This is done by

```
C = ...
ell.set_ref("C", C)
```

## Warning

The tensor introduced in `set_ref` must not be built on the fly, i.e. it must be built before using in `set_ref` and not destroyed before calculating concentration tensors. Moreover `set_ref` must be applied each time the `ellipsoid` is modified (by `shape` or `set_prop`).

Once the reference medium is set for the desired property, the strain-strain concentration tensor is obtained by

```
ell.eE
```

## Notes

- The terminology `eE` recalls that this concentration tensor relates the microscopic strain (`e` for **e**psilon) to the macroscopic strain (`E` for $\boldsymbol{E}$).
- The object returned is not a `tensor` (which is designed only for symmetric tensors) but a $6\times 6$ `numpy.ndarray` since $\mathbb{A}^E$ may not fulfill the major symmetry (i.e. not necessarily symmetric in its Kelvin-Mandel notation).
- The Hill tensor required in the concentration tensor is calculated with default parameters (analytical if the matrix is isotropic, using the `NUMINT` algorithm if the matrix is anisotropic). However it is possible to force parameters by applying `ell.set_param_eshelby(algo=NUMINT, epsroots=1.e-4, epsabs=1.e-4, epsrel=1.e-4, maxnb=100000)`.

In [ ]:
ell = ellipsoid(shape=spherical, prop={"C": stiff_kmu(5., 3.)})
C = stiff_kmu(72., 32.)
for ω in [0.1, 1.]:
    ell.shape = spheroidal(ω)
    ell.set_ref("C", C)
    A = ell.eE
    print(f"AE(ω={ω})=\n", A, "\n")
    # note that A is not major-symmetric if ω≠1

In [ ]:
ell.shape = spheroidal(0.1, math.pi/3, math.pi/4)
ell.set_prop("C", stiff_kmu(3., 2.))
ell.set_ref("C", C)
print(ell)
print(ell.eE)

## The four concentration tensors

Observing that $\boldsymbol{\sigma}=\mathbb{C}^\mathcal{I}:\boldsymbol{\varepsilon}$ within the ellipsoid and introducing the macroscopic stress tensor $\boldsymbol{\Sigma}=\mathbb{C}:\boldsymbol{E}$, all concentration tensors can be defined as

$$
\begin{aligned}
    \boldsymbol{\varepsilon}_{|\mathcal{E}_{\boldsymbol{A}}}=\mathbb{A}^E:\boldsymbol{E}&&\\
    \boldsymbol{\varepsilon}_{|\mathcal{E}_{\boldsymbol{A}}}=\mathbb{A}^\Sigma:\boldsymbol{\Sigma}&\quad\textrm{ with }\quad \mathbb{A}^\Sigma=\mathbb{A}^E:\mathbb{C}^{-1}&\\
    \boldsymbol{\sigma}_{|\mathcal{E}_{\boldsymbol{A}}}=\mathbb{B}^E:\boldsymbol{E}&\quad\textrm{ with }\quad \mathbb{B}^E=\mathbb{C}^\mathcal{I}:\mathbb{A}^E&\\
    \boldsymbol{\sigma}_{|\mathcal{E}_{\boldsymbol{A}}}=\mathbb{B}^\Sigma:\boldsymbol{\Sigma}&\quad\textrm{ with }\quad \mathbb{B}^\Sigma=\mathbb{C}^\mathcal{I}:\mathbb{A}^E:\mathbb{C}^{-1}&
\end{aligned}
$$

These concentration tensors are accessible through the attributes `eE`, `eS`, `sE` and `sS`.

In [ ]:
ell = ellipsoid(shape=spherical, prop={"C": stiff_kmu(5., 3.)})
C = stiff_kmu(72., 32.)
for ω in [0.1, 1.]:
    ell.shape = spheroidal(ω)
    ell.set_ref("C", C)
    print(f"AE(ω={ω})=\n", ell.eE, "\n")
    print(f"AS(ω={ω})=\n", ell.eS, "\n")
    print(f"BE(ω={ω})=\n", ell.sE, "\n")
    print(f"BS(ω={ω})=\n", ell.sS, "\n")

$\,$